# 16S Analyses of the Longitudinal Acne Study
## CTF and RPCA Plots

Date created: 10/18/2024

Notebook author: Britta De Pessemier 

Data analysis by: Tyler Myers, Britta De Pessemier, and Yang Chen

This notebook plots the following:
- CTF plots 
- RPCA plots

In [68]:
# Import essential Python packages
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse
from adjustText import adjust_text

# Scientific and statistical packages
import scipy.stats as ss
from skbio import DistanceMatrix
from skbio.stats.ordination import OrdinationResults
from skbio.stats.distance import permanova

# BIOM and Qiime2 related packages
import biom
from biom import load_table
from qiime2 import Artifact

# Gemelli RPCA and rarefaction utilities
from gemelli.rpca import rpca # Optional: import auto_rpca if you use auto_rpca later
from gemelli.utils import qc_rarefaction


In [69]:
# Function to draw an ellipse based on the covariance of the points
def draw_ellipse(mean, cov, ax, color, label=None, scale_factor=1.5):
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    order = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]
    
    # Calculate the angle between the x-axis and the largest eigenvector
    angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
    
    # Width and height of the ellipse (scaling applied)
    width, height = 2 * np.sqrt(eigenvalues) * scale_factor
    
    # Create the ellipse and add it to the plot
    ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
    ax.add_patch(ell)

## CTF 



In [75]:


# Define the color palette for the groups
group_colors = {
    "Healthy": "#58a35b",  # Color for Healthy
    "Acne_L": "#dd7966",   # Color for Acne Lesional
    "Acne_NL": "#6ab2bd"   # Color for Acne Non-lesional
}

# Map the old group names to new ones for the legend
group_name_mapping = {
    "Healthy": "Healthy",
    "Acne_L": "Acne Lesional",
    "Acne_NL": "Acne Non-lesional"
}

# List of folders to iterate over and corresponding titles and filenames
folders = [
    ('174950', 'V1-V3_CTF', 'CTF - 16S rRNA (V1-V3)'),
    ('174951', 'V4_CTF', 'CTF - 16S rRNA (V4)')
]

# Loop over the folders
for folder, file_suffix, plot_title in folders:
    print(f"Processing folder: {folder}")
    
    # Load the distance matrix artifact
    table = Artifact.load(f'../Data/16S/CTF/ctf-results_{folder}/distance_matrix.qza')
    
    # View as a skbio DistanceMatrix object
    distance_matrix = table.view(DistanceMatrix)
    
    # Convert the skbio DistanceMatrix to a Pandas DataFrame
    data = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)
    
    # Display the DataFrame
    print(data)
    
    # Load the subject_biplot artifact
    biplot = Artifact.load(f'../Data/16S/CTF/ctf-results_{folder}/subject_biplot.qza')
    
    # View as an OrdinationResults object
    ordination = biplot.view(OrdinationResults)
    
    # Extract the sample coordinates as a DataFrame
    sample_coordinates = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    
    # Extract the feature (or variable) coordinates as a DataFrame
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    
    # Display the DataFrames
    print("Sample Coordinates:")
    print(sample_coordinates)
    print("\nFeature Coordinates:")
    print(feature_coordinates)
    
    # Extract the proportion explained
    proportion_explained = ordination.proportion_explained
    
    # Convert to percentages for PC1 and PC2
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    print(f"PC1 explains {pc1_explained_variance:.2f}% of the variance")
    print(f"PC2 explains {pc2_explained_variance:.2f}% of the variance")
    
    # Rename the sample coordinates DataFrame to spca_df
    spca_df = sample_coordinates
    
    # Load the metadata from the subject-metadata.tsv file
    metadata_df = pd.read_csv(f'../Data/16S/CTF/ctf-results_{folder}/subject-metadata.tsv', sep='\t')
    
    # Map the index of spca_df (sample names) to the SampleID in metadata_df to get the 'group'
    spca_df["Group"] = spca_df.index.map(metadata_df.set_index("#SampleID")["group"])
    
    # Rename the columns of spca_df for clarity (PC1, PC2, PC3)
    spca_df.columns = ['PC1', 'PC2', 'PC3', 'Group']
    
    # Display the updated spca_df
    print(spca_df)
    
    # Plotting
    mm = 1/25.4
    fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
    # Create scatter plot with different colors based on the Group column
    for group_name, group in spca_df.groupby('Group'):
        sns.scatterplot(
            data=group,
            x="PC1",
            y="PC2",
            facecolor=group_colors[group_name],  # Semi-transparent fill color
            edgecolor=group_colors[group_name],  # Colored border
            alpha=0.8,  # Transparency for the fill
            linewidth=0.5,  # Thickness of the borders
            ax=ax,
            label=group_name_mapping[group_name]
        )
    
        # Calculate mean and covariance matrix for the group
        points = group[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        cov = np.cov(points, rowvar=False)
    
        # Draw the ellipse for the group (with scale factor applied)
        draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)
    
        # Add a subtle star (8-pointed) at the mean location for each group
        ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
    
    # Add axis labels and include the explained variance percentages
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)
    
    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
   # Add the title consistently in the top-left corner using axes coordinates
    ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)

    # Customize legend
    plt.legend(
        frameon=False,
        fontsize=7,
        title_fontsize=7
    )
    
    # Adjust plot style and save
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{file_suffix}.png", dpi=600)
    plt.show()


Processing folder: 174950
                   LAMI.RD017.D14.C1  LAMI.RD003.D28.C2  LAMI.RD011.D0.C1  \
LAMI.RD017.D14.C1           0.000000          15.187849         13.257771   
LAMI.RD003.D28.C2          15.187849           0.000000          5.475794   
LAMI.RD011.D0.C1           13.257771           5.475794          0.000000   
LAMI.RD011.D28.C1          13.083151           5.925958          0.674414   
LAMI.RD006.D0.C2           13.581639           6.303939          0.955279   
...                              ...                ...               ...   
LAMI.RD310.D9.C1           10.333933           6.415692          5.613001   
LAMI.RD310.D21.C1          10.123376           6.583676          5.914402   
LAMI.RD310.D0.C1           10.588279           6.113444          5.464744   
LAMI.RD310.D14.C1          10.385060           6.272099          5.531075   
LAMI.RD310.D28.C1          10.042077           6.342499          5.498209   

                   LAMI.RD011.D28.C1  LAMI.RD006.

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_47591/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_47591/3457773424.py:107: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_47591/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 

Processing folder: 174951
                   LAMI.RD011.D14.C1  LAMI.RD006.D14.C2  LAMI.RD006.D14.C1  \
LAMI.RD011.D14.C1           0.000000           6.479502           7.139171   
LAMI.RD006.D14.C2           6.479502           0.000000           0.825376   
LAMI.RD006.D14.C1           7.139171           0.825376           0.000000   
LAMI.RD018.D0.C1            6.718586           5.770834           6.472288   
LAMI.RD018.D0.C2            6.491061           5.198449           5.919783   
...                              ...                ...                ...   
LAMI.RD314.D16.C1           5.805068           7.323252           7.994251   
LAMI.RD314.D21.C1           5.700178           7.377919           8.065342   
LAMI.RD314.D0.C1            6.335841           7.256653           7.877528   
LAMI.RD314.D28.C1           5.713597           7.539581           8.218511   
LAMI.RD314.D14.C1           5.861592           7.324871           7.996571   

                   LAMI.RD018.D0.C1  

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_47591/3457773424.py:138: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## RPCA

In [14]:
# import Python packages
import pandas as pd
import numpy as np
from numpy import array
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
import scipy
import scipy.stats as ss
from skbio.stats.distance import permanova
import biom
from biom import load_table
from gemelli.rpca import rpca #import auto_rpca
from gemelli.utils import qc_rarefaction
from skbio.stats.distance import permanova, DistanceMatrix
from adjustText import adjust_text

In [78]:
# Set the random seed for reproducibility
np.random.seed(123)
# Define the color palette for the groups
group_colors = {
    "Healthy": "#58a35b",  # Color for Healthy
    "Acne_L": "#dd7966",   # Color for Acne Lesional
    "Acne_NL": "#6ab2bd"   # Color for Acne Non-lesional
}

# Map the old group names to new ones for the legend
group_name_mapping = {
    "Healthy": "Healthy",
    "Acne_L": "Acne Lesional",
    "Acne_NL": "Acne Non-lesional"
}

# Define the folder names, titles, and file paths for the two tables
datasets = [
    ('174950', 'V1-V3', '../Data/16S/RPCA/174950_rarefied_table_unannotated_absolute_abundances.biom'),
    ('174951', 'V4', '../Data/16S/RPCA/174951_rarefied_table_unannotated_absolute_abundances.biom')
]

# Loop over both tables
for dataset_id, region_title, biom_file in datasets:
    print(f"Processing dataset: {dataset_id} - {region_title}")
    
    # Load the biom table and perform RPCA
    biom_tbl = biom.load_table(biom_file)
    
    # Perform RPCA with auto rank estimation
    np.seterr(divide='ignore')
    ordination, distance = rpca(biom_tbl)

    # Read the metadata file
    metadata_file = '../Metadata/metadata_final_18062024.tsv'
    metadata_df = pd.read_csv(metadata_file, sep='\t')

    # Extract and view sample ordinations from RPCA result
    spca_df = pd.DataFrame(ordination.samples)
    spca_df["Group"] = spca_df.index.map(metadata_df.set_index("SampleID")["group"])
    
    # calculate permanova F-statistic
    pnova_res = permanova(distance, spca_df, "Group")
    p_value = pnova_res['p-value']
    
    # Plotting
    mm = 1/25.4
    fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
    # Create scatter plot with different colors based on the Group column
    for group_name, group in spca_df.groupby('Group'):
        sns.scatterplot(
            data=group,
            x="PC1",
            y="PC2",
            facecolor=group_colors[group_name],  # Semi-transparent fill color
            edgecolor=group_colors[group_name],  # Colored border
            alpha=0.8,  # Transparency for the fill
            linewidth=0.5,  # Thickness of the borders
            ax=ax,
            label=group_name_mapping[group_name]
        )
    
        # Calculate mean and covariance matrix for the group
        points = group[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        cov = np.cov(points, rowvar=False)
    
        # Draw the ellipse for the group (with scale factor applied)
        draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)
    
        # Add a subtle star (8-pointed) at the mean location for each group
        ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
    
    # Add axis labels and include the explained variance percentages
    proportion_explained = ordination.proportion_explained
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)
    
    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=7)
    
    xticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=7)
    
    # Add the title based on the region being analyzed
    ax.text(0.05, 1.10, f'RPCA - 16S rRNA ({region_title})', fontsize=8, va='top', ha='left', transform=ax.transAxes)
    
    # Add the p-value from PERMANOVA
    ax.text(-0.18, 0.27, 'PERMANOVA', fontsize=7)
    ax.text(-0.18, 0.25, f'***p-val = {p_value:.3f}', fontsize=7)

    # Customize legend in the top-right corner
    plt.legend(
        frameon=False,
        fontsize=7,
        title_fontsize=7,
        loc='upper right'
    )
    
    # Adjust plot style and save
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"../Figures/16S_Figures/{region_title}_RPCA.png", dpi=600)
    plt.show()


Processing dataset: 174950 - V1-V3


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_47591/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_47591/1111988175.py:73: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_47591/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3

Processing dataset: 174951 - V4


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_47591/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_47591/1111988175.py:73: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_47591/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3